# Chess Engine with TensorFlow

## Dataset

In [1]:
import os
# inspired by Github user Skripkon 
# https://github.com/Skripkon/chess-engine/blob/main/engines/tensorflow/train_and_predict.ipynb
# used to train models from scratch
# took around 3 hours for 20000 games

# get the game files
files = [file for file in os.listdir("new_data_2000_Elo") if file.endswith(".pgn")]

In [2]:
from chess import pgn

# load the games
def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
            
    return games

In [3]:
from tqdm import tqdm
# write all the games together 

games = []
for file in tqdm(files):
    games.extend(load_pgn(f"new_data_2000_Elo/{file}"))

100%|██████████| 30/30 [01:32<00:00,  3.10s/it]


In [4]:
len(games) # check how many 

30000

## Build & train a neural network

In [5]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from chess import Board
import tensorflow as tf

In [6]:
#translating the board into a matrix for the predition process 
def board_to_matrix(board: Board):
    matrix = np.zeros((8, 8, 12))
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[row, col, piece_type + piece_color] = 1
    return matrix

# create the inputs based off game position and next move 
def create_input_for_nn(games):
    X = []
    y = []
    for game in games:
        board = game.board()
        for move in game.mainline_moves():
            X.append(board_to_matrix(board))
            y.append(move.uci())
            board.push(move)
    return X, y

# encode all the moves
def encode_moves(moves):
    move_to_int = {move: idx for idx, move in enumerate(set(moves))}
    return [move_to_int[move] for move in moves], move_to_int

In [7]:
# create the training data
X, y = create_input_for_nn(games)
y, move_to_int = encode_moves(y)
y = np.array(y)
X = np.array(X)

In [8]:
print(len(move_to_int))

1967


In [ ]:

# train the model
model = tf.keras.Sequential([
    # Keep spatial dimensions with padding
    tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu', 
                           input_shape=(8, 8, 12)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    
    tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    
    tf.keras.layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    
    tf.keras.layers.Flatten(),    

    # More capacity before output
    tf.keras.layers.Dense(2048, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(1024, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(move_to_int), activation='softmax')
])
model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=7,
        restore_best_weights=True,
        verbose=1
    )
]

model.fit(X, y, epochs=50, validation_split=0.1, batch_size=128, callbacks=callbacks)
# save the model
model.save("HighEloBaller2000/SSMF_50EPOCHS.keras")

import pickle
# save the encoding
with open("HighEloBaller2000/move_to_int.pkl", "wb") as f:
    pickle.dump(move_to_int, f)
int_to_move = {v: k for k, v in move_to_int.items()}
with open("HighEloBaller2000/int_to_move.pkl", "wb") as f:
    pickle.dump(int_to_move, f)
# configuration 
config = {
    "epochs": 50,
    "batch_size": 128,
    "validation_split": 0.1,
    "optimizer": "Adam",
    "input_shape": (8, 8, 12),
}
# save the configurations
with open("HighEloBaller2000/train_config.json", "w") as f:
    import json
    json.dump(config, f, indent=4)



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 8, 8, 64)       │         6,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 8, 8, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2048)           │    33,556,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1967)           │     2,016,175 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,048,623 (145.14 MB)

 Trainable params: 38,047,727 (145.14 MB)

 Non-trainable params: 896 (3.50 KB)

Epoch 1/50
29081/29081 ━━━━━━━━━━━━━━━━━━━━ 4318s 148ms/step - accuracy: 0.0518 - loss: 5.1951 - val_accuracy: 0.0842 - val_loss: 4.2118
Epoch 2/50
29081/29081 ━━━━━━━━━━━━━━━━━━━━ 4295s 148ms/step - accuracy: 0.0784 - loss: 4.4087 - val_accuracy: 0.0988 - val_loss: 3.9578
Epoch 3/50
29081/29081 ━━━━━━━━━━━━━━━━━━━━ 4300s 148ms/step - accuracy: 0.0919 - loss: 4.1657 - val_accuracy: 0.1080 - val_loss: 3.8350
Epoch 4/50
29081/29081 ━━━━━━━━━━━━━━━━━━━━ 4295s 148ms/step - accuracy: 0.1011 - loss: 4.0316 - val_accuracy: 0.1126 - val_loss: 3.7649
Epoch 5/50
29081/29081 ━━━━━━━━━━━━━━━━━━━━ 4300s 148ms/step - accuracy: 0.1083 - loss: 3.9378 - val_accuracy: 0.1172 - val_loss: 3.7166
Epoch 6/50
29081/29081 ━━━━━━━━━━━━━━━━━━━━ 4296s 148ms/step - accuracy: 0.1150 - loss: 3.8659 - val_accuracy: 0.1198 - val_loss: 3.6889
Epoch 7/50
29081/29081 ━━━━━━━━━━━━━━━━━━━━ 4289s 147ms/step - accuracy: 0.1203 - loss: 3.8066 - val_accuracy: 0.1220 - val_loss: 3.6610
Epoch 8/50
29081/29081 ━━━━━━━━━━━━━━━━━━

: 